inviscid flux jacobian 

F_inviscid(Ui,Uj) = 0.5 (F(Ui) + F(Uj) - |A| * (Uj - Ui))

dF/dUi = 0.5 * (dF/dU (Ui) + |A|)

dF/dUj = 0.5 * (dF/dU (Uj) + |A|)

Functions
- compute_dFdU
- absolute_normal_jacobian (absolute values of jacobian projected to normal direction)
- roe-average states
- inviscid_flux_jacobians

Verifications
- verify compute_dFdU with eigenvalues
- verify absolute_normal_jacobian by aribatry test case, check eigenvalues
- verify compute_dFdU with compute_dFdU_fd
- verify inviscid_flux_jacobians with inviscid_flux_jacobians_fd

In [13]:
# compute_dFdU

import numpy as np
import time

def compute_dFdU(U, normal, gamma=1.4):
    """
    Compute the inviscid flux Jacobian matrix for 3D Euler equations.

    Based on equation (A.47) from Chung's Computationa Fluid Dynamics.
    Parameters
    ----------
    U : array-like, shape (5,)
        Conservative variable vector [rho, rho*u, rho*v, rho*w, rho*E]
        where:
        - rho: density
        - u, v, w: velocity components
        - E: specific total energy

    normal : array-like, shape (3,)
        Unit normal vector [n_x, n_y, n_z]

    gamma : float, optional
        Specific heat ratio (default: 1.4 for air)

    Returns
    -------
    dF_dU : ndarray, shape (5, 5)
        Flux Jacobian matrix
    """

    # Extract conservative variables
    rho = U[0]
    rho_u = U[1]
    rho_v = U[2]
    rho_w = U[3]
    rho_E = U[4]

    # Extract normal vector components
    n_x = normal[0]
    n_y = normal[1]
    n_z = normal[2]

    # Compute primitive variables
    u = rho_u / rho
    v = rho_v / rho
    w = rho_w / rho
    E = rho_E / rho

    # Compute auxiliary quantities (from A.48)
    # phi = 0.5 * (gamma - 1) * (u^2 + v^2 + w^2)
    phi = 0.5 * (gamma - 1.0) * (u**2 + v**2 + w**2)

    # V = n_x*u + n_y*v + n_z*w (normal component of velocity)
    V = n_x * u + n_y * v + n_z * w

    # a1 = gamma*E - phi
    a1 = gamma * E - phi

    # a2 = gamma - 1
    a2 = gamma - 1.0

    # a3 = gamma - 2
    a3 = gamma - 2.0

    # Initialize Jacobian matrix
    dF_dU = np.zeros((5, 5))

    # Row 1: d(rho*V)/d(U)
    dF_dU[0, 0] = 0
    dF_dU[0, 1] = n_x
    dF_dU[0, 2] = n_y
    dF_dU[0, 3] = n_z
    dF_dU[0, 4] = 0.0

    # Row 2: d(rho*u*V)/d(U)
    dF_dU[1, 0] = n_x * phi - u * V
    dF_dU[1, 1] = V - a3 * n_x * u
    dF_dU[1, 2] = n_y * u - a2 * n_x *v
    dF_dU[1, 3] = n_z * u - a2 * n_x * w
    dF_dU[1, 4] = a2 * n_x

    # Row 3: d(rho*v*V)/d(U)
    dF_dU[2, 0] = n_y * phi - v * V
    dF_dU[2, 1] = n_x * v - a2 * n_y *u
    dF_dU[2, 2] = V - a3 * n_y * v
    dF_dU[2, 3] = n_z * v - a2 * n_y * w
    dF_dU[2, 4] = a2 * n_y

    # Row 4: d(rho*w*V)/d(U)
    dF_dU[3, 0] = n_z * phi - w * V
    dF_dU[3, 1] = n_x * w - a2 * n_z *u
    dF_dU[3, 2] = n_y * w - a2 * n_z *v
    dF_dU[3, 3] =  V - a3 * n_z * w
    dF_dU[3, 4] = a2 * n_z

    # Row 5: d(rho*H*V)/d(U)
    dF_dU[4, 0] = V * (phi - a1)
    dF_dU[4, 1] = a1 * n_x - a2 * u * V
    dF_dU[4, 2] = a1 * n_y - a2 * v * V
    dF_dU[4, 3] = a1 * n_z - a2 * w * V
    dF_dU[4, 4] = gamma * V

    return dF_dU


# compute_roe_averaged_absolute_jacobian





In [14]:
gamma = 1.4
rho = 1.225        # density (kg/m^3)
u = 300.0          # x-velocity (m/s)
v = 20              # y-velocity (m/s)
w = 25              # w-velocity (m/s)
p = 101325.0       # pressure (Pa)

rhoE = p / (gamma - 1.0) + 0.5 * rho * u**2
rhou = rho * u
rhov = rho * v
rhow = rho * w

# define normal unit vector
n = np.array([1,0,0])

U = np.array([rho, rhou, rhov, rhow, rhoE])
# check A - 1D dFdU
A = compute_dFdU(U, n, gamma)

# check with eigenstructure
# A = R * Lamda * L
# R = right eigenvectors matrix
# L =  left eigenvectors matrix
# Lamda = eigenvalue matrix

# define local speed of sound 
c = np.sqrt(gamma * p / rho)
qn = u * n[0] + v * n[1] + w * n[2]
print("Eigenvalues of A: ",np.linalg.eigvals(A))
print(f"Theorectical eigenvalues of A: {qn-c}, {qn}, {qn+c}, {qn}, {qn}")

Eigenvalues of A:  [639.87203474 300.         -39.87203474 300.         300.        ]
Theorectical eigenvalues of A: -40.29399054347107, 300.0, 640.2939905434711, 300.0, 300.0


Computation of the absolute value of roe-averaged jacobian 
1. absolute value of arbitary jacobian matrix from I do Like CFD equation 3.6.16 to 3.6.26
- Verify by aribitary case of U and n
- Check eigenvalues

2. Plug in roe-averaged states to the function

Point to notes
- this formulation do not need the tangential vectors as in 2D case

In [15]:
"""
Absolute Normal Jacobian |A_n| for the 3D Euler equations.

Reference: equations 3.6.16 – 3.6.26.

Conservative variable vector:
    U = [rho, rho*u, rho*v, rho*w, rho*E]   (shape 5)

The result is assembled from three contributions (eqs. 3.6.24 – 3.6.26):

    |A_n| = |qn - c| * r1*l1^T          (eq. 3.6.25)
          + |qn|     * (r2*l2^T + r4*l4^T + r5*l5^T)   (eq. 3.6.24)
          + |qn + c| * r3*l3^T          (eq. 3.6.26)
"""


def absolute_normal_jacobian(U, n, gamma=1.4):
    """
    Compute the absolute value of the normal Jacobian |A_n| for the
    3D Euler equations without tangent-vector ambiguity (eqs. 3.6.16–3.6.26).

    Parameters
    ----------
    U : array-like, shape (5,)
        Conservative variables [rho, rho*u, rho*v, rho*w, rho*E].
    n : array-like, shape (3,)
        Face normal vector (need not be unit length; normalised internally).
    gamma : float
        Ratio of specific heats (default 1.4).

    Returns
    -------
    abs_An : ndarray, shape (5, 5)
        Absolute normal Jacobian matrix.
    """
    U = np.asarray(U, dtype=float)
    n = np.asarray(n, dtype=float)
    n = n / np.linalg.norm(n)          # unit normal

    # ------------------------------------------------------------------ #
    # Primitive / thermodynamic quantities
    # ------------------------------------------------------------------ #
    rho  = U[0]
    vel  = U[1:4] / rho                # velocity vector  v = (u, v, w)
    E    = U[4]   / rho                # total energy per unit mass
    q2   = np.dot(vel, vel)            # |v|^2
    p    = (gamma - 1.0) * rho * (E - 0.5 * q2)
    c    = np.sqrt(gamma * p / rho)    # speed of sound
    H    = E + p / rho                 # total enthalpy per unit mass
    qn   = np.dot(vel, n)              # normal velocity  q_n
    M2   = q2 / c**2                   # Mach^2
    Mn   = qn / c                      # normal Mach number  M_n
    g1   = gamma - 1.0

    # ------------------------------------------------------------------ #
    # Contribution from eigenvalue qn  (eq. 3.6.24)
    # |qn| * (r2*l2' + r4*l4' + r5*l5')
    # ------------------------------------------------------------------ #
    mid = np.zeros((5, 5))

    # row 0
    mid[0, 0]   =  1.0 - 0.5 * g1 * M2
    mid[0, 1:4] =  (g1 / c**2) * vel
    mid[0, 4]   = -(g1 / c**2)

    # rows 1-3
    mid[1:4, 0]   = -0.5 * g1 * M2 * vel + qn * n
    mid[1:4, 1:4] =  (g1 / c**2) * np.outer(vel, vel) + np.eye(3) - np.outer(n, n)
    mid[1:4, 4]   = -(g1 / c**2) * vel

    # row 4
    mid[4, 0]   =  qn**2 - 0.5 * q2 * (1.0 + 0.5 * g1 * M2)
    mid[4, 1:4] =  (1.0 + 0.5 * g1 * M2) * vel - qn * n
    mid[4, 4]   = -0.5 * g1 * M2

    # ------------------------------------------------------------------ #
    # Contribution from eigenvalue (qn - c)  (eq. 3.6.25)
    # r1 * l1'
    # ------------------------------------------------------------------ #
    l1 = np.empty(5)
    l1[0]   =  0.25 * g1 * M2 + 0.5 * Mn
    l1[1:4] = -(g1 / (2.0 * c**2)) * vel - n / (2.0 * c)
    l1[4]   =  g1 / (2.0 * c**2)

    r1 = np.empty(5)
    r1[0]   =  1.0
    r1[1:4] =  vel - c * n
    r1[4]   =  H - qn * c

    A1 = np.outer(r1, l1)

    # ------------------------------------------------------------------ #
    # Contribution from eigenvalue (qn + c)  (eq. 3.6.26)
    # r3 * l3'
    # ------------------------------------------------------------------ #
    l3 = np.empty(5)
    l3[0]   =  0.25 * g1 * M2 - 0.5 * Mn
    l3[1:4] = -(g1 / (2.0 * c**2)) * vel + n / (2.0 * c)
    l3[4]   =  g1 / (2.0 * c**2)

    r3 = np.empty(5)
    r3[0]   =  1.0
    r3[1:4] =  vel + c * n
    r3[4]   =  H + qn * c

    A3 = np.outer(r3, l3)

    # ------------------------------------------------------------------ #
    # Assemble  |A_n| = |qn-c|*A1 + |qn|*mid + |qn+c|*A3   (eq. 3.6.19)
    # ------------------------------------------------------------------ #
    abs_An = abs(qn - c) * A1 + abs(qn) * mid + abs(qn + c) * A3

    return abs_An

In [16]:
# Verification of absolute_normal_jacobian(U, n, gamma=1.4)
# Check eigenvalues of abs_An

# define testing U
rho = 1.2
vel = np.array([300.0, 50.0, -20.0])
p   = 101325.0
E   = p / ((gamma - 1) * rho) + 0.5 * np.dot(vel, vel)

U   = np.array([rho, rho*vel[0], rho*vel[1], rho*vel[2], rho*E])

# define normal unit vector
n = np.array([1.0, 0.5, 0.2])
n_unit = n / np.linalg.norm(n)          # normalize first

# compute absolute normal jacobian (absolute of jacobian projeceted to normal vector)
abs_An = absolute_normal_jacobian(U, n_unit, gamma=1.4)

# compute eigenvalues and eigenvectors
eigenvalues = np.linalg.eigvals(abs_An)

# theorectical eigenvalues of the absolute jacobian 
qn = vel @ n_unit
c = np.sqrt(gamma * p / rho)

theoretical_eigenvalues = np.array([np.abs(qn-c), qn, qn+c, qn, qn])

print("Eigenvalues of An: ",np.sort(eigenvalues))
print("Theorectical eigenvalues of An:", np.sort(theoretical_eigenvalues))


Eigenvalues of An:  [ 61.1957064  282.62474093 282.62474093 282.62474093 626.44518826]
Theorectical eigenvalues of An: [ 61.1957064  282.62474093 282.62474093 282.62474093 626.44518826]


In [17]:
# function to output roe-averages

def roe_average(UL, UR, gamma=1.4):
    """
    Compute the Roe-averaged conservative variable vector for the 3D Euler equations.
 
    Parameters
    ----------
    UL, UR : array-like, shape (5,)
        Left/right conservative vectors [rho, rho*u, rho*v, rho*w, rho*E].
    gamma : float
        Ratio of specific heats (default 1.4).
 
    Returns
    -------
    U_roe : ndarray, shape (5,)
        Roe-averaged conservative variable vector.
    """
    UL = np.asarray(UL, dtype=float)
    UR = np.asarray(UR, dtype=float)
 
    rhoL, rhoR = UL[0], UR[0]
    velL = UL[1:4] / rhoL
    velR = UR[1:4] / rhoR
    EL   = UL[4] / rhoL
    ER   = UR[4] / rhoR
 
    pL = (gamma - 1.0) * rhoL * (EL - 0.5 * np.dot(velL, velL))
    pR = (gamma - 1.0) * rhoR * (ER - 0.5 * np.dot(velR, velR))
    HL = EL + pL / rhoL    # total enthalpy per unit mass
    HR = ER + pR / rhoR
 
    wL = np.sqrt(rhoL)     # Roe weight for left state
    wR = np.sqrt(rhoR)     # Roe weight for right state
    ws = wL + wR
 
    rho_roe = wL * wR                          # = sqrt(rhoL * rhoR)
    vel_roe = (wL * velL + wR * velR) / ws
    H_roe   = (wL * HL   + wR * HR)   / ws
 
    # Recover E from H:  H = gamma*E - (gamma-1)/2 * |v|^2
    q2_roe = np.dot(vel_roe, vel_roe)
    E_roe  = (H_roe + (gamma - 1.0) * 0.5 * q2_roe) / gamma
 
    U_roe = np.empty(5)
    U_roe[0]   = rho_roe
    U_roe[1:4] = rho_roe * vel_roe
    U_roe[4]   = rho_roe * E_roe
    
    return U_roe

In [18]:
# inviscid_flux_jacobians

def inviscid_flux_jacobians(U_i, U_j, n,gamma=1.4):
    """
    Compute the normal inviscid flux Jacobian blocks at a cell face between
    cell i (left/owner) and cell j (right/neighbor).

    Parameters
    ----------
    U_i, U_j : array_like, shape (5,)
        Conservative variables [rho, rho*u, rho*v, rho*w, rho*E] of the
        two cells sharing the face.
    n : array_like, shape (3,)
        Unit normal vector [nx, ny, nz] of the face, pointing from i to j.
    gamma: float
        Ratio of specific heats
    Returns
    -------
    d(F)/d(U_i)  -- Inviscid Jacobian wrt the left-cell conservative state.
    d(F)/d(U_j)  -- Inviscid Jacobian wrt the right-cell conservative state.
    """

    U_roe = roe_average(U_i, U_j, gamma)
    abs_A_roe = absolute_normal_jacobian(U_roe, n, gamma=1.4)
    dFdUi = 0.5 * (compute_dFdU(U_i, n) + abs_A_roe)
    dFdUj = 0.5 * (compute_dFdU(U_j, n) - abs_A_roe)
    
    return dFdUi, dFdUj

Verification with Finite Difference
- define euler_flux function
- define compute_dFdU_fd: dF/dU by finite differencing
- define inviscid_flux_jacobians_fd: dF/dUi, dF/dUj by finite differencing

In [19]:
def euler_flux(U, n, gamma=1.4):
    """
    Computes the Euler flux projected onto normal n. (Euler normal flux)
    U  = [rho, rhou, rhov, rhow, rhoE]
    n  = [nx, ny, nz]  (need not be unit)
    F  = [rho*un, rhou*un + P*nx, rhov*un + P*ny, rhow*un + P*nz, (rhoE+P)*un]
    un = u*nx + v*ny + w*nz
    """
    rho  = U[0]
    rhou = U[1]; rhov = U[2]; rhow = U[3]; rhoE = U[4]

    u = rhou / rho
    v = rhov / rho
    w = rhow / rho

    nx, ny, nz = n[0], n[1], n[2]
    un = u * nx + v * ny + w * nz                         # fix 1

    P = (gamma - 1.0) * (rhoE - 0.5 * rho * (u**2 + v**2 + w**2))  # fix 2

    F = np.zeros(5)                                       # fix 3
    F[0] = rho  * un
    F[1] = rhou * un + P * nx
    F[2] = rhov * un + P * ny
    F[3] = rhow * un + P * nz
    F[4] = (rhoE + P) * un

    return F


def compute_dFdU_fd(U, n, eps, gamma=1.4):
    """
    Computes the 3D Euler Flux Jacobian dF/dU by forward finite differences.

    Parameters
    ----------
    U     : ndarray (5,)  [rho, rhou, rhov, rhow, rhoE]
    n     : ndarray (3,)  face normal vector
    eps   : float         perturbation step size
    gamma : float         ratio of specific heats

    Returns
    -------
    dFdU : ndarray (5, 5)
    """
    F_base = euler_flux(U, n, gamma)
    dFdU   = np.zeros((5, 5))
    I      = np.eye(5)

    for j in range(5):
        Up = U + eps * I[:, j]
        dFdU[:, j] = (euler_flux(Up, n, gamma) - F_base) / eps

    return dFdU

# inviscid_flux_jacobians by finite difference

def inviscid_flux_jacobians_fd(U_i, U_j, n, eps, gamma=1.4):
    """
    Compute the normal inviscid flux Jacobian blocks at a cell face between
    cell i (left/owner) and cell j (right/neighbor).

    Parameters
    ----------
    U_i, U_j : array_like, shape (5,)
        Conservative variables [rho, rho*u, rho*v, rho*w, rho*E] of the
        two cells sharing the face.
    n : array_like, shape (3,)
        Unit normal vector [nx, ny, nz] of the face, pointing from i to j.
    gamma: float
        Ratio of specific heats
    Returns
    -------
    d(F)/d(U_i)  -- Inviscid Jacobian wrt the left-cell conservative state.
    d(F)/d(U_j)  -- Inviscid Jacobian wrt the right-cell conservative state.
    """
    U_roe     = roe_average(U_i, U_j, gamma)
    abs_A_roe = absolute_normal_jacobian(U_roe, n, gamma=gamma)
    dFdUi = 0.5 * (compute_dFdU_fd(U_i, n, eps, gamma) + abs_A_roe)
    dFdUj = 0.5 * (compute_dFdU_fd(U_j, n, eps, gamma) - abs_A_roe)
    return dFdUi, dFdUj

In [20]:
"""
Tests verifying inviscid_flux_jacobians_fd against inviscid_flux_jacobians,
and compute_dFdU_fd against compute_dFdU.

Assumes the following are already defined in the notebook:
    euler_flux, compute_dFdU, compute_dFdU_fd,
    roe_average, absolute_normal_jacobian,
    inviscid_flux_jacobians, inviscid_flux_jacobians_fd

All test normals are unit vectors so euler_flux (unscaled) and
absolute_normal_jacobian (internally normalised) stay consistent.
Tolerances are relative — Jacobian entries span many orders of magnitude.
"""
# ------------------------------------------------------------------
# Utilities
# ------------------------------------------------------------------

def _rel_err(A, B):
    """Max relative error  max|A-B| / (max|B| + 1e-30).
    Safe when B is large; breaks down when B ~ 0 (use _rel_err_pair instead)."""
    return np.max(np.abs(A - B)) / (np.max(np.abs(B)) + 1e-30)


def _rel_err_pair(Ai, Aj, Bi, Bj):
    """Relative error for a Jacobian pair, normalised by the shared scale
    max(|Bi|, |Bj|).  Safe even when one block is identically zero
    (e.g. fully-upwinded supersonic case where dF/dU_j = 0 exactly)."""
    
    scale = max(np.max(np.abs(Bi)), np.max(np.abs(Bj))) + 1e-30
    ei = np.max(np.abs(Ai - Bi)) / scale
    ej = np.max(np.abs(Aj - Bj)) / scale
    return ei, ej


def _make_state(rho, u, v, w, p, gamma=1.4):
    E = p / ((gamma - 1) * rho) + 0.5 * (u**2 + v**2 + w**2)
    return np.array([rho, rho*u, rho*v, rho*w, rho*E])


def _unit(v):
    v = np.asarray(v, float)
    return v / np.linalg.norm(v)


# ------------------------------------------------------------------
# Test states and normals
# ------------------------------------------------------------------
gamma = 1.4

U_sub1 = _make_state(1.2, 100., 30., -10., 101325., gamma)
U_sub2 = _make_state(0.9,  80., 20.,   5.,  90000., gamma)
U_sup1 = _make_state(1.0, 800., 50.,   0.,  50000., gamma)
U_sup2 = _make_state(1.5, 750., 30.,  10.,  40000., gamma)

_normals = {
    "x-axis"  : _unit([1., 0.,  0.]),
    "y-axis"  : _unit([0., 1.,  0.]),
    "oblique" : _unit([1., 0.5, 0.2]),
    "diagonal": _unit([1., 1.,  1.]),
}

eps  = 1e-8
rtol = 1e-3    # forward FD is O(eps); entries O(1e7) -> abs err O(10-1e4)


print("eps: ", eps)
# ==================================================================
# Test 0 — compute_dFdU_fd vs compute_dFdU (flux Jacobian baseline)
# ==================================================================
print("=" * 62)
print("Test 0: compute_dFdU_fd vs compute_dFdU (flux Jacobian)")
print("=" * 62)
for name, n in _normals.items():
    J_fd  = compute_dFdU_fd(U_sub1, n, eps, gamma)
    J_ref = compute_dFdU(U_sub1, n, gamma)
    err = _rel_err(J_fd, J_ref)
    ok  = "PASS" if err < rtol else "FAIL"
    print(f"  [{ok}] n={name:10s}  rel_err={err:.2e}")


# ==================================================================
# Test 1 — inviscid_flux_jacobians_fd vs inviscid_flux_jacobians
#           (general subsonic states, multiple normals)
# ==================================================================
print()
print("=" * 62)
print("Test 1: FD vs analytical (general subsonic states)")
print("=" * 62)
for name, n in _normals.items():
    dFdUi_fd,  dFdUj_fd  = inviscid_flux_jacobians_fd(U_sub1, U_sub2, n, eps, gamma)
    dFdUi_ref, dFdUj_ref = inviscid_flux_jacobians(U_sub1, U_sub2, n, gamma)
    ei, ej = _rel_err_pair(dFdUi_fd, dFdUj_fd, dFdUi_ref, dFdUj_ref)
    ok = "PASS" if max(ei, ej) < rtol else "FAIL"
    print(f"  [{ok}] n={name:10s}  rel_err_i={ei:.2e}  rel_err_j={ej:.2e}")


# ==================================================================
# Test 2 — Identical states (U_i = U_j = U)
# ==================================================================
# Subsonic: dF/dU_j is small but non-zero.
# Supersonic: all eigenvalues positive -> |A_n| = A_n ->
#   dF/dU_j = 0.5*(A_n - |A_n|) = 0  (fully upwinded, one block is zero).
# _rel_err_pair normalises both blocks by the same scale so the zero
# block does not cause division-by-near-zero.
print()
print("=" * 62)
print("Test 2: Identical states (U_i = U_j)")
print("=" * 62)
n = _normals["oblique"]
for label, U in [("subsonic", U_sub1), ("supersonic", U_sup1)]:
    dFdUi_fd,  dFdUj_fd  = inviscid_flux_jacobians_fd(U, U, n, eps, gamma)
    dFdUi_ref, dFdUj_ref = inviscid_flux_jacobians(U, U, n, gamma)
    ei, ej = _rel_err_pair(dFdUi_fd, dFdUj_fd, dFdUi_ref, dFdUj_ref)
    ok = "PASS" if max(ei, ej) < rtol else "FAIL"
    print(f"  [{ok}] {label:12s}  rel_err_i={ei:.2e}  rel_err_j={ej:.2e}")


# ==================================================================
# Test 3 — Supersonic states (U_i != U_j)
# ==================================================================
print()
print("=" * 62)
print("Test 3: Supersonic states (U_i != U_j)")
print("=" * 62)
n = _normals["oblique"]
dFdUi_fd,  dFdUj_fd  = inviscid_flux_jacobians_fd(U_sup1, U_sup2, n, eps, gamma)
dFdUi_ref, dFdUj_ref = inviscid_flux_jacobians(U_sup1, U_sup2, n, gamma)
ei, ej = _rel_err_pair(dFdUi_fd, dFdUj_fd, dFdUi_ref, dFdUj_ref)
ok = "PASS" if max(ei, ej) < rtol else "FAIL"
print(f"  [{ok}] rel_err_i={ei:.2e}  rel_err_j={ej:.2e}")


# ==================================================================
# Test 4 — eps sensitivity (FD error decreases linearly with eps)
# ==================================================================
print()
print("=" * 62)
print("Test 4: eps sensitivity (forward FD is O(eps), ratio ~0.10 per decade)")
print("=" * 62)
n = _normals["x-axis"]
dFdUi_ref, dFdUj_ref = inviscid_flux_jacobians(U_sub1, U_sub2, n, gamma)
prev_err = None
for eps_test in [1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8]:
    dFdUi_fd, dFdUj_fd = inviscid_flux_jacobians_fd(U_sub1, U_sub2, n, eps_test, gamma)
    ei, ej = _rel_err_pair(dFdUi_fd, dFdUj_fd, dFdUi_ref, dFdUj_ref)
    err = max(ei, ej)
    ratio_str = f"ratio={err/prev_err:.2f}" if prev_err else ""
    print(f"  eps={eps_test:.0e}  rel_err={err:.3e}  {ratio_str}")
    prev_err = err


# ==================================================================
# Test 5 — Face conservation (swap states + flip normal)
# ==================================================================
# |A_roe(-n)| = |A_roe(n)|  (negating eigenvalues then |.| cancels),
# so flipping n alone does NOT negate the blocks.
# The correct anti-symmetry is (U_i, U_j, n) -> (U_j, U_i, -n):
#
#   dF(U_j, U_i, -n)/d(U_j) = -dF(U_i, U_j, n)/d(U_j)
#   dF(U_j, U_i, -n)/d(U_i) = -dF(U_i, U_j, n)/d(U_i)
#
# Checked on both the FD and the analytical versions.
print()
print("=" * 62)
print("Test 5: Face conservation (swap states + flip normal)")
print("=" * 62)
n = _normals["oblique"]
for tag, fn in [("fd", inviscid_flux_jacobians_fd), ("analytical", inviscid_flux_jacobians)]:
    kw  = dict(eps=eps) if tag == "fd" else {}
    pos = fn(U_sub1, U_sub2,  n, **kw, gamma=gamma)
    neg = fn(U_sub2, U_sub1, -n, **kw, gamma=gamma)
    dFdUi_pos, dFdUj_pos = pos
    dFdUi_neg, dFdUj_neg = neg
    ei, ej = _rel_err_pair(dFdUi_neg, dFdUj_neg, -dFdUj_pos, -dFdUi_pos)
    ok = "PASS" if max(ei, ej) < rtol else "FAIL"
    print(f"  [{ok}] {tag:12s}  rel_err_i={ei:.2e}  rel_err_j={ej:.2e}")

eps:  1e-08
Test 0: compute_dFdU_fd vs compute_dFdU (flux Jacobian)
  [PASS] n=x-axis      rel_err=1.53e-08
  [PASS] n=y-axis      rel_err=1.49e-08
  [PASS] n=oblique     rel_err=4.78e-08
  [PASS] n=diagonal    rel_err=1.15e-08

Test 1: FD vs analytical (general subsonic states)
  [PASS] n=x-axis      rel_err_i=1.19e-08  rel_err_j=9.60e-09
  [PASS] n=y-axis      rel_err_i=1.62e-08  rel_err_j=1.31e-08
  [PASS] n=oblique     rel_err_i=3.73e-08  rel_err_j=1.51e-08
  [PASS] n=diagonal    rel_err_i=9.70e-09  rel_err_j=1.88e-08

Test 2: Identical states (U_i = U_j)
  [PASS] subsonic      rel_err_i=3.61e-08  rel_err_j=3.61e-08
  [PASS] supersonic    rel_err_i=8.13e-09  rel_err_j=8.13e-09

Test 3: Supersonic states (U_i != U_j)
  [PASS] rel_err_i=8.98e-09  rel_err_j=4.73e-09

Test 4: eps sensitivity (forward FD is O(eps), ratio ~0.10 per decade)
  eps=1e-03  rel_err=8.127e-04  
  eps=1e-04  rel_err=8.135e-05  ratio=0.10
  eps=1e-05  rel_err=8.136e-06  ratio=0.10
  eps=1e-06  rel_err=8.135e-07 

Check the improvement in run time for finite differenncing
- Assume the function is called for N times while assembling the global flux jacobian
- N = Number of cells * 5

In [22]:
# check the improvement in run time for finite differenncing

gamma = 1.4
U_i = _make_state(1.2, 100., 30., -10., 101325., gamma)
U_j = _make_state(0.9,  80., 20.,   5.,  90000., gamma)
n   = _unit([1., 0.5, 0.2])
eps = 1e-8
N   = 20000   # number of repeated calls (e.g if cells = 3060, called = 3060 * 5 ~ 20000)

# ── Analytical ──────────────────────────────────────────────────────
t0 = time.perf_counter()
for _ in range(N):
    inviscid_flux_jacobians(U_i, U_j, n, gamma)
t_analytical = (time.perf_counter() - t0) / N * 1e6   # µs per call

# ── Finite difference ───────────────────────────────────────────────
t0 = time.perf_counter()
for _ in range(N):
    inviscid_flux_jacobians_fd(U_i, U_j, n, eps, gamma)
t_fd = (time.perf_counter() - t0) / N * 1e6           # µs per call

print(f"Number of calls N: {N}")
print(f"Analytical  : {t_analytical:.2f} µs/call")
print(f"FD          : {t_fd:.2f} µs/call")
print(f"Ratio of run time (FD/Analytical)    : {t_fd / t_analytical:.3f}")

Number of calls N: 20000
Analytical  : 98.02 µs/call
FD          : 166.03 µs/call
Ratio of run time (FD/Analytical)    : 1.694
